In [1]:
import pandas as pd
import numpy as np
import sqlite3
import joblib
import os
import warnings
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                      cross_val_predict, train_test_split)
from sklearn.metrics import roc_auc_score, classification_report, f1_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

os.makedirs("../output/improved", exist_ok=True)
print("All imports successful!")

All imports successful!


In [2]:
conn = sqlite3.connect("../data/attrition.db")
df = pd.read_sql("SELECT * FROM employees", conn)
conn.close()

label_cols = ["BusinessTravel", "Department", "EducationField",
              "Gender", "JobRole", "MaritalStatus", "OverTime"]
le = LabelEncoder()
df_model = df.copy()
for col in label_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

role_avg = df_model.groupby("JobRole")["MonthlyIncome"].transform("mean")
df_model["SalaryToRoleRatio"]     = (df_model["MonthlyIncome"] / role_avg).round(3)
df_model["TenureStabilityScore"]  = (df_model["YearsAtCompany"] /
                                      (df_model["TotalWorkingYears"] + 1)).round(3)
df_model["SatisfactionComposite"] = (df_model["JobSatisfaction"] +
                                      df_model["EnvironmentSatisfaction"] +
                                      df_model["WorkLifeBalance"]) / 3

drop_cols = [c for c in ["Attrition", "Attrition_Binary", "EmployeeNumber",
                          "OverTime_Binary"] if c in df_model.columns]

# Load Boruta-selected features from Notebook 04
final_features = pd.read_csv("../outputs/final_features.csv",
                              header=None)[0].tolist()
final_features = [f for f in final_features
                  if f in df_model.drop(columns=drop_cols).columns]

X = df_model.drop(columns=drop_cols)[final_features]
y = df_model["Attrition_Binary"]

print(f"Features: {len(final_features)} — {final_features}")
print(f"Dataset:  {X.shape}")
print(f"Attrition rate: {y.mean():.3f}  ({y.sum()} positives / {len(y)} total)")

Features: 7 — ['Age', 'MonthlyIncome', 'OverTime', 'TotalWorkingYears', 'SalaryToRoleRatio', 'SatisfactionComposite', 'DailyRate']
Dataset:  (1470, 7)
Attrition rate: 0.161  (237 positives / 1470 total)


In [3]:
print("=== FIX 1: 10-FOLD STRATIFIED CV ===")
print("Previously used: single 80/20 holdout (47 positives in test)")
print("Now using: 10-fold stratified CV (23 positives per fold average)\n")

# Use 10 folds instead of 5 — more stable on small positive class
skf_10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
skf_5  = StratifiedKFold(n_splits=5,  shuffle=True, random_state=42)

# Baseline comparison — same LightGBM, different CV strategy
lgbm_base = LGBMClassifier(
    n_estimators=200, class_weight="balanced",
    random_state=42, n_jobs=-1, verbose=-1
)

auc_5fold  = cross_val_score(lgbm_base, X, y, cv=skf_5,  scoring="roc_auc").mean()
auc_10fold = cross_val_score(lgbm_base, X, y, cv=skf_10, scoring="roc_auc").mean()

print(f"Same model — 5-fold CV AUC:   {auc_5fold:.4f}")
print(f"Same model — 10-fold CV AUC:  {auc_10fold:.4f}")
print(f"\nUsing 10-fold for all remaining evaluations.")

=== FIX 1: 10-FOLD STRATIFIED CV ===
Previously used: single 80/20 holdout (47 positives in test)
Now using: 10-fold stratified CV (23 positives per fold average)

Same model — 5-fold CV AUC:   0.7165
Same model — 10-fold CV AUC:  0.7157

Using 10-fold for all remaining evaluations.


In [4]:
print("\n=== FIX 2: REDUCED COMPLEXITY + RETUNING ===")

# --- LightGBM: lower max_depth, stricter regularisation ---
def obj_lgbm(trial):
    p = {
        "n_estimators":      trial.suggest_int("n_estimators", 50, 300),
        "max_depth":         trial.suggest_int("max_depth", 2, 5),      # was 3-10
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 8, 40),    # was 20-100
        "subsample":         trial.suggest_float("subsample", 0.5, 0.9),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 0.9),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 80),
        "reg_alpha":         trial.suggest_float("reg_alpha", 0.0, 2.0),   # L1
        "reg_lambda":        trial.suggest_float("reg_lambda", 0.0, 2.0),  # L2
        "class_weight": "balanced",
        "random_state": 42, "n_jobs": -1, "verbose": -1
    }
    return cross_val_score(
        LGBMClassifier(**p), X, y, cv=skf_10, scoring="roc_auc"
    ).mean()

study_lgbm = optuna.create_study(direction="maximize")
study_lgbm.optimize(obj_lgbm, n_trials=80, show_progress_bar=True)
print(f"LightGBM best 10-fold AUC: {study_lgbm.best_value:.4f}")
print(f"Best params: {study_lgbm.best_params}")


=== FIX 2: REDUCED COMPLEXITY + RETUNING ===


  0%|          | 0/80 [00:00<?, ?it/s]

LightGBM best 10-fold AUC: 0.7680
Best params: {'n_estimators': 118, 'max_depth': 4, 'learning_rate': 0.013583018868030079, 'num_leaves': 13, 'subsample': 0.886659050006241, 'colsample_bytree': 0.519900031440691, 'min_child_samples': 80, 'reg_alpha': 0.49163237065961174, 'reg_lambda': 1.5066812989517848}


In [5]:
def obj_gb(trial):
    p = {
        "n_estimators":     trial.suggest_int("n_estimators", 50, 250),
        "max_depth":        trial.suggest_int("max_depth", 2, 4),       # was 2-6
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 0.9),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 30),
        "max_features":     trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "random_state": 42
    }
    return cross_val_score(
        GradientBoostingClassifier(**p), X, y, cv=skf_10, scoring="roc_auc"
    ).mean()

study_gb = optuna.create_study(direction="maximize")
study_gb.optimize(obj_gb, n_trials=50, show_progress_bar=True)
print(f"GradientBoosting best 10-fold AUC: {study_gb.best_value:.4f}")

  0%|          | 0/50 [00:00<?, ?it/s]

GradientBoosting best 10-fold AUC: 0.7736


In [6]:
def obj_rf(trial):
    p = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 400),
        "max_depth":        trial.suggest_int("max_depth", 3, 8),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 15),
        "max_features":     trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "class_weight": "balanced",
        "random_state": 42, "n_jobs": -1
    }
    return cross_val_score(
        RandomForestClassifier(**p), X, y, cv=skf_10, scoring="roc_auc"
    ).mean()

study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(obj_rf, n_trials=50, show_progress_bar=True)
print(f"RandomForest best 10-fold AUC: {study_rf.best_value:.4f}")

  0%|          | 0/50 [00:00<?, ?it/s]

RandomForest best 10-fold AUC: 0.7678


In [7]:
print("\n=== FIX 3: class_weight INSTEAD OF SMOTE + CLEAN OOF BLEND ===")

best_lgbm = LGBMClassifier(
    **study_lgbm.best_params,
    class_weight="balanced",       # no SMOTE — handles imbalance natively
    random_state=42, n_jobs=-1, verbose=-1
)
best_gb = GradientBoostingClassifier(
    **study_gb.best_params,
    random_state=42
)
best_rf = RandomForestClassifier(
    **study_rf.best_params,
    class_weight="balanced",
    random_state=42, n_jobs=-1
)

X_np = X.values
y_np = y.values

# Generate clean OOF predictions — no SMOTE leakage
def get_oof(model, X, y, cv):
    oof = np.zeros(len(y))
    for train_idx, val_idx in cv.split(X, y):
        clone = type(model)(**model.get_params())
        clone.fit(X[train_idx], y[train_idx])
        oof[val_idx] = clone.predict_proba(X[val_idx])[:, 1]
    return oof

print("Generating OOF predictions (10-fold, no SMOTE)...")
oof_lgbm = get_oof(best_lgbm, X_np, y_np, skf_10)
oof_gb   = get_oof(best_gb,   X_np, y_np, skf_10)
oof_rf   = get_oof(best_rf,   X_np, y_np, skf_10)

auc_lgbm = roc_auc_score(y_np, oof_lgbm)
auc_gb   = roc_auc_score(y_np, oof_gb)
auc_rf   = roc_auc_score(y_np, oof_rf)

print(f"\nOOF AUCs (10-fold, class_weight, no SMOTE):")
print(f"  LightGBM:        {auc_lgbm:.4f}")
print(f"  GradientBoosting:{auc_gb:.4f}")
print(f"  RandomForest:    {auc_rf:.4f}")

# AUC-weighted blend
total = auc_lgbm + auc_gb + auc_rf
w_lgbm = auc_lgbm / total
w_gb   = auc_gb   / total
w_rf   = auc_rf   / total

oof_blend = w_lgbm*oof_lgbm + w_gb*oof_gb + w_rf*oof_rf
blend_auc = roc_auc_score(y_np, oof_blend)

print(f"\nBlend weights — LGBM:{w_lgbm:.3f}  GB:{w_gb:.3f}  RF:{w_rf:.3f}")
print(f"OOF Blend AUC:   {blend_auc:.4f}")


=== FIX 3: class_weight INSTEAD OF SMOTE + CLEAN OOF BLEND ===
Generating OOF predictions (10-fold, no SMOTE)...

OOF AUCs (10-fold, class_weight, no SMOTE):
  LightGBM:        0.7678
  GradientBoosting:0.7728
  RandomForest:    0.7671

Blend weights — LGBM:0.333  GB:0.335  RF:0.332
OOF Blend AUC:   0.7712


In [8]:
print("\n=== OLD vs NEW COMPARISON ===")

comparison = pd.DataFrame({
    "Model": [
        "Notebook 04 — OOF blend (old)",
        "Notebook 05 — Holdout (old)",
        "Notebook 06 — LightGBM (fixed)",
        "Notebook 06 — GradBoost (fixed)",
        "Notebook 06 — RandomForest (fixed)",
        "Notebook 06 — Blend (fixed)"
    ],
    "AUC": [
        0.7744, 0.6600,
        auc_lgbm, auc_gb, auc_rf, blend_auc
    ],
    "Evaluation": [
        "5-fold OOF", "Single holdout",
        "10-fold OOF", "10-fold OOF",
        "10-fold OOF", "10-fold OOF"
    ],
    "SMOTE": [
        "Yes", "Yes", "No", "No", "No", "No"
    ]
})
comparison = comparison.sort_values("AUC", ascending=False)
print(comparison.to_string(index=False))
comparison.to_csv("../output/improved/old_vs_new_comparison.csv", index=False)

# Chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#534AB7" if "06" in m else "#D3D1C7"
          for m in comparison["Model"]]
bars = ax.barh(comparison["Model"], comparison["AUC"], color=colors)
ax.axvline(0.5, color="red",   ls="--", alpha=0.4, label="Random (0.5)")
ax.axvline(0.75, color="green", ls="--", alpha=0.4, label="Good (0.75)")
for bar, val in zip(bars, comparison["AUC"]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=10)
ax.set_xlabel("AUC")
ax.set_title("Old vs New — AUC comparison")
ax.legend(); ax.grid(True, alpha=0.2, axis="x")
ax.set_xlim(0.4, 1.0)
plt.tight_layout()
plt.savefig("../output/improved/old_vs_new_chart.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nSaved old_vs_new_chart.png")


=== OLD vs NEW COMPARISON ===
                             Model      AUC     Evaluation SMOTE
     Notebook 04 — OOF blend (old) 0.774400     5-fold OOF   Yes
   Notebook 06 — GradBoost (fixed) 0.772809    10-fold OOF    No
       Notebook 06 — Blend (fixed) 0.771194    10-fold OOF    No
    Notebook 06 — LightGBM (fixed) 0.767755    10-fold OOF    No
Notebook 06 — RandomForest (fixed) 0.767077    10-fold OOF    No
       Notebook 05 — Holdout (old) 0.660000 Single holdout   Yes

Saved old_vs_new_chart.png


In [9]:
print("\n=== FINAL MODEL + THRESHOLD OPTIMIZATION ===")

# Fit final models on full data
best_lgbm.fit(X_np, y_np)
best_gb.fit(X_np, y_np)
best_rf.fit(X_np, y_np)

# Final blend probabilities
final_proba = (w_lgbm * best_lgbm.predict_proba(X_np)[:, 1] +
               w_gb   * best_gb.predict_proba(X_np)[:, 1] +
               w_rf   * best_rf.predict_proba(X_np)[:, 1])

# Business cost threshold sweep
COST_FN = 0.50
COST_FP = 0.02
avg_annual = df["MonthlyIncome"].mean() * 12

records = []
for t in np.arange(0.05, 0.95, 0.01):
    y_pred_t = (final_proba >= t).astype(int)
    fn = ((y_pred_t == 0) & (y_np == 1)).sum()
    fp = ((y_pred_t == 1) & (y_np == 0)).sum()
    cost = fn * avg_annual * COST_FN + fp * avg_annual * COST_FP
    records.append({
        "threshold": round(t, 2),
        "cost":      round(cost, 0),
        "f1":        round(f1_score(y_np, y_pred_t, zero_division=0), 4)
    })

thresh_df = pd.DataFrame(records)
best_t    = thresh_df.loc[thresh_df["cost"].idxmin(), "threshold"]
default_cost = thresh_df[thresh_df["threshold"]==0.50]["cost"].values[0]
optimal_cost = thresh_df[thresh_df["threshold"]==best_t]["cost"].values[0]

print(f"Optimal threshold: {best_t}")
print(f"Cost saving vs default 0.5: ${default_cost - optimal_cost:,.0f}")

# Save models and outputs
joblib.dump(best_lgbm, "../output/improved/lgbm_final.pkl")
joblib.dump(best_gb,   "../output/improved/gb_final.pkl")
joblib.dump(best_rf,   "../output/improved/rf_final.pkl")

blend_weights = {"lgbm": w_lgbm, "gb": w_gb, "rf": w_rf,
                 "threshold": best_t}
joblib.dump(blend_weights, "../output/improved/blend_weights.pkl")

thresh_df.to_csv("../output/improved/threshold_analysis.csv", index=False)

print("\nAll models saved to output/improved/")


=== FINAL MODEL + THRESHOLD OPTIMIZATION ===
Optimal threshold: 0.24
Cost saving vs default 0.5: $3,566,208

All models saved to output/improved/


In [10]:
print("\n" + "="*55)
print("NOTEBOOK 06 — FINAL SUMMARY")
print("="*55)

print(f"""
What changed from Notebook 04/05:
  Fix 1: 5-fold → 10-fold CV      more stable AUC on small data
  Fix 2: Dropped TabNet            was dragging blend down (0.60)
  Fix 3: SMOTE → class_weight      removes synthetic data leakage

Results:
  LightGBM  10-fold OOF AUC: {auc_lgbm:.4f}
  GradBoost 10-fold OOF AUC: {auc_gb:.4f}
  RandomFor 10-fold OOF AUC: {auc_rf:.4f}
  Blend     10-fold OOF AUC: {blend_auc:.4f}

  Previous best (Notebook 04): 0.7744
  New best (Notebook 06):      {blend_auc:.4f}
  Improvement:                 {blend_auc - 0.7744:+.4f}

Resume bullet:
  Built 3-model blending ensemble (LightGBM + GradientBoosting
  + RandomForest) tuned with Optuna — 10-fold OOF AUC {blend_auc:.2f},
  with business cost-optimized threshold saving ${default_cost-optimal_cost:,.0f}
  vs default classifier.
""")

print("Output files:")
for f in sorted(os.listdir("../output/improved")):
    print(f"  {f}")


NOTEBOOK 06 — FINAL SUMMARY

What changed from Notebook 04/05:
  Fix 1: 5-fold → 10-fold CV      more stable AUC on small data
  Fix 2: Dropped TabNet            was dragging blend down (0.60)
  Fix 3: SMOTE → class_weight      removes synthetic data leakage

Results:
  LightGBM  10-fold OOF AUC: 0.7678
  GradBoost 10-fold OOF AUC: 0.7728
  RandomFor 10-fold OOF AUC: 0.7671
  Blend     10-fold OOF AUC: 0.7712

  Previous best (Notebook 04): 0.7744
  New best (Notebook 06):      0.7712
  Improvement:                 -0.0032

Resume bullet:
  Built 3-model blending ensemble (LightGBM + GradientBoosting
  + RandomForest) tuned with Optuna — 10-fold OOF AUC 0.77,
  with business cost-optimized threshold saving $3,566,208
  vs default classifier.

Output files:
  blend_weights.pkl
  gb_final.pkl
  lgbm_final.pkl
  old_vs_new_chart.png
  old_vs_new_comparison.csv
  rf_final.pkl
  threshold_analysis.csv
